In [3]:
import numpy as np
import pandas as pd
import folium
from sklearn.neighbors import BallTree
import requests
from tqdm import tqdm
import time

#### 데이터 가져오기

In [4]:
DATA_PATH = 'D:/workspaces/00_Project/crawlProject/project/data/'

In [5]:
def get_all_data(data_list):
    '''
    Put all data in dictionary
    '''
    dfs = {}

    for data_name in data_list:
        df = pd.read_csv(DATA_PATH + f"{data_name}/df_{data_name}.csv")
        dfs[data_name] = df
    return dfs

In [6]:
def get_df_grid(data_name):
    df_grid = pd.read_csv(DATA_PATH + f'grid/{data_name}.csv')
    return df_grid

In [25]:
data_list = ['hospital',
             'factory',
             'bridge',
             'substation',
             'core']

dfs = get_all_data(data_list)
df_grid = get_df_grid('grid_50m_4351cells')

#### 가중치 설정함수

In [8]:
def set_score(df, alpha):
    df['score'] = [alpha] * len(df) 

In [21]:
set_score(dfs['hospital'], 3)
set_score(dfs['factory'], 2)
set_score(dfs['substation'], 3)
set_score(dfs['core'], 5)
set_score(dfs['bridge'], 2)

#### 점수 산출

In [10]:
def building_cover(coords_grid, coords_building, RANGE_KM=1):
    """
    격자 좌표를 기준으로 반경 내 포함되는 건물 정보를 계산하는 함수

    Parameters
    ----------
    coords_grid : array-like (n, 2)
        격자 중심 좌표 (위도, 경도, degree)

    coords_building : array-like (m, 2)
        건물 좌표 (위도, 경도, degree)

    range_km : float, optional
        탐색 반경 (km 단위), 기본값 1km

    Returns
    -------
    df_result : pandas.DataFrame
        각 격자별 포함된 건물 개수와 인덱스 정보를 담은 데이터프레임
        - grid_id : 격자 인덱스
        - building_count : 포함된 건물 개수
        - building_indices : 포함된 건물 인덱스 리스트
    """

    # rad 변환
    grid_rad = np.deg2rad(coords_grid)
    building_rad = np.deg2rad(coords_building)

    # BallTree (건물 기준)
    tree = BallTree(building_rad, metric='haversine')

    # 반경 (km → rad)
    radius = RANGE_KM / 6371

    # 각 grid 기준 건물 찾기
    indices = tree.query_radius(grid_rad, r=radius)

    # 결과 정리
    building_indices = indices
    building_count = [len(idx) for idx in building_indices]

    df_result = pd.DataFrame({
        'grid_id': range(len(coords_grid)),
        'building_count': building_count,
        'building_indices': building_indices,
    })

    return df_result

In [11]:
def calc_score(dfs, RANGE_KM=1.0):
    '''
    격자 점수 산출
    dfs : 데이터 딕셔너리
    RANGE_KM : 레이더 사정거리
    '''

    df_building = pd.concat(dfs.values(), ignore_index=True)

    coords_building = df_building[['latitude', 'longitude']].values
    coords_grid = df_grid[['center_lat', 'center_lng']].values


    # 점수 산정
    df_result = building_cover(coords_grid, coords_building, RANGE_KM)

    scores = []
    for cover in df_result['building_indices'].values:
        sc = 0
        for grid_no in cover:
            sc += df_building.iloc[grid_no]['score']
        scores.append(sc)

    df_result['score'] = scores
    return df_result

In [ ]:
RANGE_KM = 1.5
df_result = calc_score(dfs, 1.5)

# 최고 점수 격자(인덱스) 선정 -> 지금은 1순위만 선정함
max_score = df_result['score'].max()
best_points = list(df_result[df_result['score'] == max_score].index )

if len(best_points)

In [ ]:
if len(best_points) != 1:
    print('d')
else:
    pos

d


In [27]:
best_points

[4134, 4204, 4346]

In [28]:
# MAP 객체 생성
m = folium.Map(location=[37.5, 127.04], zoom_start=7)

# 격자 경계점
grid_lat_min = df_grid["sw_lat"].min()
grid_lat_max = df_grid["ne_lat"].max()
grid_lon_min = df_grid["sw_lng"].min()
grid_lon_max = df_grid["ne_lng"].max()


# 격자 구역 표시
folium.Rectangle(
    bounds=[[grid_lat_min, grid_lon_min], [grid_lat_max, grid_lon_max]],
    color="blue",
    weight=2,
    fill=True,
    fill_opacity=0.05,
    tooltip="격자 전체 영역"
).add_to(m)

ICON_MAP = {
    "hospital":   folium.Icon(color="red",     icon="hospital",        prefix="fa"),
    "factory":    folium.Icon(color="blue",    icon="industry",        prefix="fa"),
    "substation": folium.Icon(color="green",   icon="bolt",            prefix="fa"),
    "bridge":     folium.Icon(color="purple",  icon="road",            prefix="fa"),
    "core":       folium.Icon(color="darkred", icon="broadcast-tower", prefix="fa"),
}


for key, df in dfs.items():

    layer = folium.FeatureGroup(name=key)

    filtered = df[
        (df['latitude'] >= grid_lat_min) &
        (df['latitude'] <= grid_lat_max) &
        (df['longitude'] >= grid_lon_min) &
        (df['longitude'] <= grid_lon_max)
    ].copy()

    for _, row in filtered.iterrows():
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            tooltip=row['name'],
            popup=folium.Popup(row['name'], max_width=200),
            icon=ICON_MAP.get(key, folium.Icon(color="gray", icon="question", prefix="fa"))
        ).add_to(layer)

    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)  # 레이어 패널 기본으로 열어둠

# 레이더 위치 표시
for point in best_points:
    loca = df_grid.iloc[point][['center_lat','center_lng']].values
    folium.Marker(location=loca).add_to(m)
    folium.Circle(location=loca, 
                  radius=RANGE_KM*1000,
                  color='red',
                  fill=True,
                  fill_color='red',
                  fill_opacity=0.2).add_to(m)

m.save("map.html")
print("저장 완료: map.html")

저장 완료: map.html


#### BallTree 방법

In [23]:
def make_cover_df(coords, range_km=3, include_self=False):
    """
    coords: (n,2) array [lat, lng] (degree)
    range_km: 커버 반경 (km)
    include_self: 자기 자신 포함 여부
    """

    if len(coords) == 0:
        return pd.DataFrame()

    # degree → rad
    coords_rad = np.deg2rad(coords)

    # BallTree 생성
    tree = BallTree(coords_rad, metric='haversine')

    # km → rad
    radius = range_km / 6371

    # 이웃 찾기
    indices = tree.query_radius(coords_rad, r=radius)

    # 자기 자신 처리
    if include_self:
        cover_indices = indices
    else:
        cover_indices = [idx[idx != i] for i, idx in enumerate(indices)]

    # 커버 개수
    cover_count = [len(idx) for idx in cover_indices]

    # 결과
    df_coverage = pd.DataFrame({
        'jammer_id': range(len(coords)),
        'cover_count': cover_count,
        'covered_points': cover_indices
    })

    return df_coverage

In [35]:
coords = df_grid[['center_lat', 'center_lng']].values
make_cover_df(coords, range_km=1)

,jammer_id,cover_count,covered_points
0,0,315,"[26, 1, 2, 3, 186, 155, 58, 7, 8, 57, 77, 76, ..."
1,1,327,"[26, 2, 3, 186, 155, 58, 7, 8, 57, 77, 76, 56,..."
2,2,341,"[26, 1, 3, 186, 155, 58, 7, 8, 57, 77, 76, 56,..."
3,3,354,"[26, 1, 2, 186, 155, 58, 7, 8, 57, 77, 76, 56,..."
4,4,366,"[26, 1, 2, 3, 186, 155, 58, 7, 8, 57, 77, 76, ..."
...,...,...,...
4323,4323,494,"[3896, 3838, 3952, 4056, 4101, 3781, 3954, 383..."
4324,4324,500,"[3952, 4056, 4101, 3781, 3954, 3839, 3782, 389..."
4325,4325,418,"[3895, 3720, 3896, 3838, 3719, 3779, 3778, 395..."
4326,4326,428,"[3895, 3720, 3896, 3838, 3779, 3778, 3952, 383..."
